# Notas — Aula 16: Capstone — configurar, montar e testar o robô

Marco: o modelo de features ganha uma terceira dimensão (tipo de grade),
uma função monta o robô inteiro a partir de um dicionário de
configuração, e `pytest` entra formalmente — fixtures, `assert`,
`@pytest.mark.parametrize` — testando o contrato desse robô configurado.


In [1]:
from enum import Enum

LADO_GRADE = 10

class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)


class EstrategiaPadrao:
    def mover(self, robo):
        return robo.avancar()


class EstrategiaZigzag:
    def __init__(self, periodo=2):
        self.periodo = periodo
        self.passos_dados = 0

    def mover(self, robo):
        if self.passos_dados > 0 and self.passos_dados % self.periodo == 0:
            lado = "DIR" if (self.passos_dados // self.periodo) % 2 else "ESQ"
            robo.girar(lado)
        moveu = robo.avancar()
        if moveu:
            self.passos_dados += 1
        return moveu


class Robo:
    LADO_GRADE = 10
    _registro = {}

    def __init_subclass__(cls, categoria="geral", **kwargs):
        super().__init_subclass__(**kwargs)
        Robo._registro[cls.__name__] = cls
        cls.categoria = categoria

    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE, obstaculos=None, estrategia=None):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.obstaculos = obstaculos if obstaculos is not None else {}
        self.estrategia = estrategia if estrategia is not None else EstrategiaPadrao()

    def sensor_frente(self):
        dx, dy = self.direcao.value
        nx, ny = self.x + dx, self.y + dy
        return (0 <= nx < Robo.LADO_GRADE and 0 <= ny < Robo.LADO_GRADE
                and (nx, ny) not in self.obstaculos)

    def avancar(self):
        if self.sensor_frente():
            dx, dy = self.direcao.value
            self.x += dx
            self.y += dy
            return True
        return False

    def girar(self, lado):
        ordem = [Direcao.LESTE, Direcao.NORTE, Direcao.OESTE, Direcao.SUL]
        if lado == "ESQ":
            self.direcao = ordem[(ordem.index(self.direcao) + 1) % 4]
        elif lado == "DIR":
            self.direcao = ordem[(ordem.index(self.direcao) - 1) % 4]

    def mover(self):
        return self.estrategia.mover(self)


class RoboExplorador(Robo, categoria="reconhecimento"):
    pass


class RoboBlindado(Robo, categoria="defensivo"):
    pass


def criar_robo(tipo_nome, nome, **kwargs):
    classe = Robo._registro.get(tipo_nome)
    if classe is None:
        raise ValueError(f"tipo desconhecido: {tipo_nome!r}")
    return classe(nome, **kwargs)


ESTRATEGIAS_VALIDAS = {"padrao", "zigzag"}
FABRICA_ESTRATEGIAS = {"padrao": EstrategiaPadrao, "zigzag": EstrategiaZigzag}

EXCLUI = {"RoboBlindado": {"zigzag"}}


class ConfiguracaoInvalida(Exception):
    pass


def validar_configuracao_lps(tipo_nome, estrategia_nome):
    if tipo_nome not in Robo._registro:
        raise ConfiguracaoInvalida(f"tipo desconhecido: {tipo_nome!r}")
    if estrategia_nome not in ESTRATEGIAS_VALIDAS:
        raise ConfiguracaoInvalida(f"estratégia desconhecida: {estrategia_nome!r}")
    if estrategia_nome in EXCLUI.get(tipo_nome, set()):
        raise ConfiguracaoInvalida(f"{tipo_nome} exclui a estratégia {estrategia_nome!r}")


## Terceira dimensão do modelo: tipo de grade

O modelo de features da Aula 15 tinha duas alternativas (tipo, estratégia). Hoje
entra a terceira: cada preset de grade é uma **função sem argumento** que devolve
um dicionário de `obstaculos` novo a cada chamada — nunca um dicionário já pronto
compartilhado entre instâncias.


In [2]:
def grade_vazia():
    return {}


def grade_moldura():
    borda = ([(x, 0) for x in range(10)] + [(x, 9) for x in range(10)] +
             [(0, y) for y in range(10)] + [(9, y) for y in range(10)])
    return {pos: True for pos in borda}


FABRICA_GRADES = {
    "vazia": grade_vazia,
    "moldura": grade_moldura,
}

print(len(grade_vazia()), len(grade_moldura()))


0 36


### Sua vez

Complete `grade_labirinto()`: duas paredes internas — todas as casas `(3, y)` com
`y` de 0 a 6, e todas as casas `(6, y)` com `y` de 3 a 9. Acrescente ao
`FABRICA_GRADES` e recalcule `GRADES_VALIDAS`.

*Dica: `[(3, y) for y in range(0, 7)] + [(6, y) for y in range(3, 10)]`, depois um
dict comprehension como as outras grades.*


In [3]:
# TODO: complete grade_labirinto() e registre em FABRICA_GRADES
def grade_labirinto():
    return {}


FABRICA_GRADES["labirinto"] = grade_labirinto
GRADES_VALIDAS = set(FABRICA_GRADES)

print(len(grade_labirinto()))


0


## Configuração por dicionário

Configuração de verdade chega como **dado**, não como argumentos nomeados
digitados um por um. `validar_configuracao` e `criar_robo_configurado` ganham a
grade; `montar_robo_de_config` faz a ponte de um dicionário até um robô pronto.


In [4]:
def validar_configuracao(tipo_nome, estrategia_nome, grade_nome="vazia"):
    validar_configuracao_lps(tipo_nome, estrategia_nome)
    if grade_nome not in GRADES_VALIDAS:
        raise ConfiguracaoInvalida(f"grade desconhecida: {grade_nome!r}")


def criar_robo_configurado(tipo_nome, nome, estrategia_nome="padrao", grade_nome="vazia",
                            **kwargs):
    validar_configuracao(tipo_nome, estrategia_nome, grade_nome)
    obstaculos = FABRICA_GRADES[grade_nome]()
    robo = criar_robo(tipo_nome, nome, obstaculos=obstaculos, **kwargs)
    robo.estrategia = FABRICA_ESTRATEGIAS[estrategia_nome]()
    return robo


def montar_robo_de_config(config):
    return criar_robo_configurado(
        config["tipo_nome"], config["nome"],
        estrategia_nome=config.get("estrategia_nome", "padrao"),
        grade_nome=config.get("grade_nome", "vazia"),
        x=config.get("x", 0), y=config.get("y", 0),
    )


robo1 = montar_robo_de_config({
    "tipo_nome": "RoboExplorador", "nome": "Scout",
    "estrategia_nome": "zigzag", "grade_nome": "labirinto", "x": 1, "y": 1,
})
print(robo1.nome, len(robo1.obstaculos))


Scout 0


### Sua vez

Complete `montar_frota_de_config(configs)`: uma lista de dicionários vira uma lista
de robôs, um por item, cada um passando por `montar_robo_de_config`.

*Dica: uma linha, list comprehension.*


In [5]:
# TODO: uma linha — list comprehension chamando montar_robo_de_config por item
def montar_frota_de_config(configs):
    return []


frota = montar_frota_de_config([
    {"tipo_nome": "RoboExplorador", "nome": "a", "estrategia_nome": "zigzag"},
    {"tipo_nome": "RoboBlindado", "nome": "b"},
])
print([r.nome for r in frota])


[]


## Antes do pytest: o jeito manual, nomeado

`checar_invariante` e `verificar_invariante_frota` já são
testes — só sem framework por trás: uma função chama, compara, acumula falhas.


In [6]:
def checar_configuracoes(casos):
    falhas = []
    for tipo_nome, estrategia_nome, esperado_valido in casos:
        try:
            criar_robo_configurado(tipo_nome, "Teste", estrategia_nome=estrategia_nome)
            passou = True
        except ConfiguracaoInvalida:
            passou = False
        if passou != esperado_valido:
            falhas.append((tipo_nome, estrategia_nome))
    return falhas


falhas = checar_configuracoes([
    ("RoboBlindado", "zigzag", False),
    ("RoboExplorador", "zigzag", True),
])
print(falhas)


[]


### Sua vez

Complete `checar_configuracoes` de um jeito diferente: em vez de devolver só o
par `(tipo_nome, estrategia_nome)` de cada falha, devolva uma string explicando o
que se esperava vs. o que aconteceu — ex.: `"RoboBlindado+zigzag: esperava válido,
mas não é"`.

*Dica: monte a string dentro do `if passou != esperado_valido`, com um f-string
diferente pra cada direção do erro (esperava válido vs. esperava inválido).*


In [7]:
# TODO: monte uma mensagem descritiva em vez de só (tipo_nome, estrategia_nome)
def checar_configuracoes_detalhado(casos):
    falhas = []
    for tipo_nome, estrategia_nome, esperado_valido in casos:
        try:
            criar_robo_configurado(tipo_nome, "Teste", estrategia_nome=estrategia_nome)
            passou = True
        except ConfiguracaoInvalida:
            passou = False
        if passou != esperado_valido:
            ...
    return falhas


print(checar_configuracoes_detalhado([("RoboBlindado", "zigzag", True)]))


[]


## `pytest` de verdade: fixture, `assert`, e o contrato testado

O mesmo teste de cima, agora com a ferramenta real. `pytest` roda um `test_*.py`
como **processo separado** — ele não enxerga as variáveis do notebook, só o que
o arquivo importar. Por isso o primeiro passo é gravar o modelo num arquivo `.py`
de verdade; o segundo é o arquivo de teste **importando** dele.


In [8]:
%%writefile capstone_model.py
from enum import Enum


class Direcao(Enum):
    LESTE = (1, 0)
    NORTE = (0, 1)
    OESTE = (-1, 0)
    SUL = (0, -1)


class EstrategiaPadrao:
    def mover(self, robo):
        return robo.avancar()


class EstrategiaZigzag:
    def __init__(self, periodo=2):
        self.periodo = periodo
        self.passos_dados = 0

    def mover(self, robo):
        if self.passos_dados > 0 and self.passos_dados % self.periodo == 0:
            lado = "DIR" if (self.passos_dados // self.periodo) % 2 else "ESQ"
            robo.girar(lado)
        moveu = robo.avancar()
        if moveu:
            self.passos_dados += 1
        return moveu


class Robo:
    LADO_GRADE = 10
    _registro = {}

    def __init_subclass__(cls, categoria="geral", **kwargs):
        super().__init_subclass__(**kwargs)
        Robo._registro[cls.__name__] = cls
        cls.categoria = categoria

    def __init__(self, nome, x=0, y=0, direcao=Direcao.LESTE, obstaculos=None, estrategia=None):
        self.nome = nome
        self.x = x
        self.y = y
        self.direcao = direcao
        self.obstaculos = obstaculos if obstaculos is not None else {}
        self.estrategia = estrategia if estrategia is not None else EstrategiaPadrao()

    def sensor_frente(self):
        dx, dy = self.direcao.value
        nx, ny = self.x + dx, self.y + dy
        return (0 <= nx < Robo.LADO_GRADE and 0 <= ny < Robo.LADO_GRADE
                and (nx, ny) not in self.obstaculos)

    def avancar(self):
        if self.sensor_frente():
            dx, dy = self.direcao.value
            self.x += dx
            self.y += dy
            return True
        return False

    def girar(self, lado):
        ordem = [Direcao.LESTE, Direcao.NORTE, Direcao.OESTE, Direcao.SUL]
        if lado == "ESQ":
            self.direcao = ordem[(ordem.index(self.direcao) + 1) % 4]
        elif lado == "DIR":
            self.direcao = ordem[(ordem.index(self.direcao) - 1) % 4]

    def mover(self):
        return self.estrategia.mover(self)


class RoboExplorador(Robo, categoria="reconhecimento"):
    pass


class RoboBlindado(Robo, categoria="defensivo"):
    pass


def criar_robo(tipo_nome, nome, **kwargs):
    classe = Robo._registro.get(tipo_nome)
    if classe is None:
        raise ValueError(f"tipo desconhecido: {tipo_nome!r}")
    return classe(nome, **kwargs)


ESTRATEGIAS_VALIDAS = {"padrao", "zigzag"}
FABRICA_ESTRATEGIAS = {"padrao": EstrategiaPadrao, "zigzag": EstrategiaZigzag}
EXCLUI = {"RoboBlindado": {"zigzag"}}


class ConfiguracaoInvalida(Exception):
    pass


def validar_configuracao(tipo_nome, estrategia_nome):
    if tipo_nome not in Robo._registro:
        raise ConfiguracaoInvalida(f"tipo desconhecido: {tipo_nome!r}")
    if estrategia_nome not in ESTRATEGIAS_VALIDAS:
        raise ConfiguracaoInvalida(f"estratégia desconhecida: {estrategia_nome!r}")
    if estrategia_nome in EXCLUI.get(tipo_nome, set()):
        raise ConfiguracaoInvalida(f"{tipo_nome} exclui a estratégia {estrategia_nome!r}")


def criar_robo_configurado(tipo_nome, nome, estrategia_nome="padrao", **kwargs):
    validar_configuracao(tipo_nome, estrategia_nome)
    robo = criar_robo(tipo_nome, nome, **kwargs)
    robo.estrategia = FABRICA_ESTRATEGIAS[estrategia_nome]()
    return robo


Overwriting capstone_model.py


In [9]:
%%writefile test_capstone.py
import pytest

from capstone_model import criar_robo_configurado, ConfiguracaoInvalida


def test_explorador_com_zigzag_e_valido():
    robo = criar_robo_configurado("RoboExplorador", "Scout", estrategia_nome="zigzag")
    assert robo.nome == "Scout"


@pytest.fixture
def scout():
    return criar_robo_configurado("RoboExplorador", "Scout", estrategia_nome="zigzag")


def test_scout_comeca_na_origem(scout):
    assert scout.x == 0 and scout.y == 0


@pytest.mark.parametrize("tipo_nome,estrategia_nome,valido", [
    ("RoboBlindado", "zigzag", False),
    ("RoboExplorador", "zigzag", True),
])
def test_contrato_de_configuracao(tipo_nome, estrategia_nome, valido):
    if valido:
        criar_robo_configurado(tipo_nome, "Teste", estrategia_nome=estrategia_nome)
    else:
        with pytest.raises(ConfiguracaoInvalida):
            criar_robo_configurado(tipo_nome, "Teste", estrategia_nome=estrategia_nome)


Overwriting test_capstone.py


In [10]:
!pytest test_capstone.py -v


============================= test session starts ==============================
platform darwin -- Python 3.14.6, pytest-9.1.1, pluggy-1.6.0 -- /opt/homebrew/opt/python@3.14/bin/python3.14
cachedir: .pytest_cache
rootdir: /Users/leopoldo/Documents/cin/teaching/residencia-python-alunos/notas
collecting ... 
collected 4 items                                                              

test_capstone.py::test_explorador_com_zigzag_e_valido PASSED             [ 25%]
test_capstone.py::test_scout_comeca_na_origem PASSED                     [ 50%]
test_capstone.py::test_contrato_de_configuracao[RoboBlindado-zigzag-False] PASSED [ 75%]
test_capstone.py::test_contrato_de_configuracao[RoboExplorador-zigzag-True] PASSED [100%]

============================== 4 passed in 0.01s ===============================


## Para aprofundar

- pytest — "Get Started" (documentação oficial): https://docs.pytest.org/en/stable/getting-started.html
- Fixtures (documentação oficial): https://docs.pytest.org/en/stable/how-to/fixtures.html
- `@pytest.mark.parametrize` (documentação oficial): https://docs.pytest.org/en/stable/how-to/parametrize.html
- `FeatureIDE` (ferramenta gráfica de modelagem de features): https://featureide.github.io/
- Biblioteca `transitions` (máquina de estados pronta para Python): https://github.com/pytransitions/transitions
